In [0]:
#Get AQE Parameters
# spark.conf.get("spark.sql.adaptive.enabled")
# spark.conf.get("spark.sql.adaptive.skewJoin.enabled")
# spark.conf.get("spark.sql.adaptive.coalescePartitions.enabled")
# spark.conf.get("spark.sql.autoBroadcastJoinThreshold")
# spark.conf.get("spark.sql.optimizer.dynamicPartitionPruning.enabled")

In [0]:
#Disable AQE
spark.conf.set("spark.sql.adaptive.enabled","false")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled","false")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled","false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold",-1)
spark.conf.set("spark.sql.optimizer.dynamicPartitionPruning.enabled","false")

In [0]:
#Enable AQE
# spark.conf.set("spark.sql.adaptive.enabled","true")
# spark.conf.set("spark.sql.adaptive.skewJoin.enabled","true")
# spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled","true")
# #spark.conf.set("spark.sql.autoBroadcastJoinThreshold",10485760b)
# spark.conf.set("spark.sql.optimizer.dynamicPartitionPruning.enabled","true")

In [0]:
path =  "abfss://labdata@dbdeltalakestorage.dfs.core.windows.net/invoices/invoices_201_99457.parquet"
df = spark.read.parquet(path)

In [0]:
df.repartition(288).write.mode("overwrite").partitionBy("category").saveAsTable("deltacatalog.deltadb.optimize_ex2")

In [0]:
%%time
df_ex2 = spark.read.table("deltacatalog.deltadb.optimize_ex2")
df_out = df_ex2.where("category='Clothing'").collect()

In [0]:
df.repartition(288).write.mode("overwrite").partitionBy("category").option("optimizeWrite","True").saveAsTable("deltacatalog.deltadb.optimize_ex3")

In [0]:
%%time
df_ex3 = spark.read.table("deltacatalog.deltadb.optimize_ex3")
df_out = df_ex3.where("category='Clothing'").collect()

How to Solve Different Small File Problems:

1. 
2.
3. Small FIles due to frequent writes in delta lake and size also not very big in writes, Solution : AutoCompaction

#### AutoCompaction

In [0]:
%sql
ALTER TABLE deltacatalog.deltadb.optimize_ex3 SET TBLPROPERTIES (delta.autoOptimize.optimizeWrite = 'false');
ALTER TABLE deltacatalog.deltadb.optimize_ex3 SET TBLPROPERTIES (delta.autoOptimize.autoCompact = 'true');

In [0]:
spark.conf.get("spark.databricks.delta.autoCompact.minNumFiles")

In [0]:
df_detergents = df.filter(df.category == "Clothing").withColumn("category",F.lit("Detergents"))

In [0]:
df_detergents.repartition(5).write.mode("overwrite").partitionBy("category").saveAsTable("deltacatlog.deltadb.optimize_ex3")